## Summary — Data Cleaning & Load (Python Layer)

**Libraries used**
Pandas, os, SQLAlchemy, python-dotenv

**Initial inspection**
- Loaded raw CSV with `utf-8-sig` encoding (fixed a BOM artifact that was corrupting the `Age` column name)
- Checked shape, first 5 rows, and column dtypes/null counts via `.info()`

**Column triage**
Used `.unique()` / `.nunique()` on each column to bucket into:
- **Dropped** — `EmployeeCount`, `StandardHours`, `Over18` (constant, single value across all rows — no analytical value)
- **Kept as-is** — remaining numeric and text categorical columns, verified clean (no typos, inconsistent casing, or stray whitespace)
- **Relabeled** — `Education`, `EnvironmentSatisfaction`, `JobInvolvement`, `JobSatisfaction`, `PerformanceRating`, `RelationshipSatisfaction`, `WorkLifeBalance` — mapped numeric codes to text categories per IBM's official data dictionary
- **Flagged for feature engineering (later phase)** — `YearsAtCompany`, `YearsInCurrentRole`, `YearsSinceLastPromotion`, `YearsWithCurrManager`, `TotalWorkingYears` → will feed into `tenure_years`, `tenure_band`, `attrition_risk`

**Data quality checks**
- No null values (1470/1470 non-null across all columns)
- No duplicate rows; `EmployeeNumber` confirmed as a unique identifier

**Output**
- Saved cleaned dataframe to `data/processed/HR-Employee-Attrition-cleaned.csv`
- Loaded into MySQL as Bronze table (`hr_attrition.bronze_hr_attrition`, 1470 rows) via SQLAlchemy engine

In [1]:
# importing files

import pandas as pd
import os
import sqlalchemy
import dotenv

In [9]:
# Reading file

RAW_PATH = os.path.join('..', 'data', 'raw' , 'HR-Employee-Attrition.csv')

df = pd.read_csv(RAW_PATH, encoding='utf-8-sig')
print('Shape', df.shape)



Shape (1470, 35)


In [8]:
# preview first 5 rows
 
df.head(5)

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [9]:
# Check dtype and null counts

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

In [10]:
# Check unique values to categorize Relabel, Drop and keep as is columns

print(df['EmployeeCount'].unique())
print(df['StandardHours'].unique())
print(df['Education'].unique())
print(df["Over18"].unique())
print(df['EmployeeNumber'].nunique())
print(df['BusinessTravel'].unique())
print(df['Department'].unique())
print(df['Gender'].unique())
print(df['EducationField'].unique())
print(df['JobRole'].unique())
print(df['MaritalStatus'].unique())
print(df['OverTime'].unique())
print(df['Attrition'].unique())

[1]
[80]
[2 1 4 3 5]
['Y']
1470
['Travel_Rarely' 'Travel_Frequently' 'Non-Travel']
['Sales' 'Research & Development' 'Human Resources']
['Female' 'Male']
['Life Sciences' 'Other' 'Medical' 'Marketing' 'Technical Degree'
 'Human Resources']
['Sales Executive' 'Research Scientist' 'Laboratory Technician'
 'Manufacturing Director' 'Healthcare Representative' 'Manager'
 'Sales Representative' 'Research Director' 'Human Resources']
['Single' 'Married' 'Divorced']
['Yes' 'No']
['Yes' 'No']


## Column Triage

Based on inspection of unique values and non-null counts:

**Dropped (constant/no analytical value):**
EmployeeCount, StandardHours, Over18

**Kept as-is:**
EmployeeNumber, Age, StockOptionLevel, BusinessTravel, Department, Gender,
EducationField, JobRole, MaritalStatus, OverTime, Attrition, DailyRate,
HourlyRate, MonthlyIncome, MonthlyRate, DistanceFromHome, NumCompaniesWorked,
PercentSalaryHike, TrainingTimesLastYear, JobLevel

**Relabeled (numeric codes → text, per IBM data dictionary):**
Education, EnvironmentSatisfaction, JobInvolvement, JobSatisfaction,
PerformanceRating, RelationshipSatisfaction, WorkLifeBalance

**Flagged for feature engineering (later phase):**
YearsAtCompany, YearsInCurrentRole, YearsSinceLastPromotion,
YearsWithCurrManager, TotalWorkingYears → tenure_years, tenure_band, attrition_risk

In [11]:
# Drop columns

df = df.drop(columns=['EmployeeCount', 'StandardHours', 'Over18'])



In [17]:
# Check shape

print(df.shape)

(1470, 32)


In [18]:
# Relabeled columns

education_map = {1: 'Below College', 2: 'College', 3: 'Bachelor', 4: 'Master', 5: 'Doctor'}
df['Education'] = df['Education'].map(education_map)

environmentsatisifaction_map = {1:"Low", 2:"Medium", 3:"High", 4:"Very High"} 
df['EnvironmentSatisfaction'] = df['EnvironmentSatisfaction'].map(environmentsatisifaction_map)

jobinvolvement_map = {1:"Low", 2:"Medium", 3:"High", 4:"Very High"} 
df['JobInvolvement'] = df['JobInvolvement'].map(jobinvolvement_map)

jobsatisfaction_map = {1:"Low", 2:"Medium", 3:"High", 4:"Very High"}
df['JobSatisfaction'] = df['JobSatisfaction'].map(jobsatisfaction_map)

performancerating_map = {1:"Low", 2:"Good", 3:"Excellent", 4:"Outstanding"}
df['PerformanceRating'] = df['PerformanceRating'].map(performancerating_map)

relationshipsatisfaction_map = {1:"Low", 2:"Medium", 3:"High", 4:"Very High"}
df['RelationshipSatisfaction'] = df['RelationshipSatisfaction'].map(relationshipsatisfaction_map)

workLifebalance_map = {1:"Bad", 2:"Good", 3:"Better", 4:"Best"}
df['WorkLifeBalance'] = df['WorkLifeBalance'].map(workLifebalance_map)

In [20]:
# Check relabel columns

df[['Education', 'EnvironmentSatisfaction', 'JobInvolvement', 'JobSatisfaction',
    'PerformanceRating', 'RelationshipSatisfaction', 'WorkLifeBalance']].head()

,Education,EnvironmentSatisfaction,JobInvolvement,JobSatisfaction,PerformanceRating,RelationshipSatisfaction,WorkLifeBalance
0,College,Medium,High,Very High,Excellent,Low,Bad
1,Below College,High,Medium,Medium,Outstanding,Very High,Better
2,College,Very High,Medium,High,Excellent,Medium,Better
3,Master,Very High,High,High,Excellent,High,Better
4,Below College,Low,High,Medium,Excellent,Very High,Better


In [23]:
# Check for duplicates

print(df.duplicated().sum())
print(df['EmployeeNumber'].duplicated().sum())

0
0


In [ ]:
# Check column name
df.columns



Index(['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department',
       'DistanceFromHome', 'Education', 'EducationField', 'EmployeeNumber',
       'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement',
       'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus',
       'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'OverTime',
       'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction',
       'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear',
       'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole',
       'YearsSinceLastPromotion', 'YearsWithCurrManager'],
      dtype='object')

In [ ]:
# Check column dtype

df.dtypes

Age                          int64
Attrition                   object
BusinessTravel              object
DailyRate                    int64
Department                  object
DistanceFromHome             int64
Education                   object
EducationField              object
EmployeeNumber               int64
EnvironmentSatisfaction     object
Gender                      object
HourlyRate                   int64
JobInvolvement              object
JobLevel                     int64
JobRole                     object
JobSatisfaction             object
MaritalStatus               object
MonthlyIncome                int64
MonthlyRate                  int64
NumCompaniesWorked           int64
OverTime                    object
PercentSalaryHike            int64
PerformanceRating           object
RelationshipSatisfaction    object
StockOptionLevel             int64
TotalWorkingYears            int64
TrainingTimesLastYear        int64
WorkLifeBalance             object
YearsAtCompany      

In [30]:
# Save processed output

OUTPUT_PATH = os.path.join('..', 'data', 'processed' , 'HR-Employee-Attrition-cleaned.csv')

df.to_csv(OUTPUT_PATH, index=False)
print('Saved cleaned CSV to:', OUTPUT_PATH)
print('Final shape:', df.shape)

Saved cleaned CSV to: ..\data\processed\HR-Employee-Attrition-cleaned.csv
Final shape: (1470, 32)


In [2]:
# Ensure libraries are in the same environment

import sys
!{sys.executable} -m pip install sqlalchemy python-dotenv pymysql


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: C:\Users\rohit\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
import sqlalchemy
import dotenv
print(sqlalchemy.__version__)


2.0.51


## Exporting data to MySQL

In [7]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

host = os.getenv("MYSQL_HOST")
user = os.getenv("MYSQL_USER")
paasword = os.getenv("MYSQL_PASSWORD")
database = os.getenv("MYSQL_DATABASE")


connection_string = f"mysql+pymysql://{user}:{password}@{host}/{database}"
engine = create_engine(connection_string)


print("Connection Successful")

Connection Successful


In [10]:
# Load data into MySQL

df.to_sql(
    name = 'bronze_hr_attrition',
    con = engine,
    if_exists= 'replace',
    index= False

)

print('Data loaded successfully')

Data loaded successfully
